# Детекция фейковых новостей с использованием больших языковых моделей (LLM)

В данном ноутбуке реализованы два подхода к детекции фейковых новостей на основе больших языковых моделей:

1. **Zero-shot классификация** — использование предобученной мультиязычной модели NLI (Natural Language Inference) для классификации без какого-либо дообучения на целевом датасете.
2. **Дообучение генеративной LLM с LoRA (Low-Rank Adaptation)** — параметрически эффективное дообучение (PEFT) русскоязычной GPT-модели `ai-forever/rugpt3small_based_on_gpt2` для бинарной классификации.

Оба подхода оцениваются на том же тестовом наборе, что и RuBERT и классические модели (TF-IDF, Word2Vec), для объективного сравнения.

In [ ]:
!pip install peft accelerate -q

## 1. Конфигурация и зависимости

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, asdict
from pathlib import Path
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
@dataclass
class Config:
    # Данные
    data_path:      str   = '../../data/ready_dataset.csv'
    text_column:    str   = 'combined_text'
    label_column:   str   = 'label'

    # Модели
    gpt_model_name: str   = 'ai-forever/rugpt3small_based_on_gpt2'
    nli_model_name: str   = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'
    max_length:     int   = 256
    num_labels:     int   = 2

    # LoRA
    lora_r:         int   = 16
    lora_alpha:     int   = 32
    lora_dropout:   float = 0.1

    # Обучение
    batch_size:     int   = 8
    epochs:         int   = 5
    learning_rate:  float = 2e-4
    weight_decay:   float = 0.01
    warmup_ratio:   float = 0.1
    grad_clip:      float = 1.0
    label_smoothing:float = 0.1
    patience:       int   = 2

    # Сплит (идентичен RuBERT для честного сравнения)
    test_size:      float = 0.1
    val_size:       float = 0.1
    random_state:   int   = 42
    num_workers:    int   = 0

    # Пути сохранения
    output_dir:     str   = '../../models/llm'

config = Config()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
print(f'\nGPT model:  {config.gpt_model_name}')
print(f'NLI model:  {config.nli_model_name}')

## 2. Загрузка и подготовка данных

Используется тот же датасет и та же стратегия разбиения (80/10/10, `random_state=42`, стратификация по меткам), что и при обучении RuBERT — это гарантирует справедливость сравнения.

In [ ]:
def load_and_split(cfg):
    df = pd.read_csv(cfg.data_path)
    for col in [cfg.text_column, cfg.label_column]:
        if col not in df.columns:
            raise ValueError(f'Колонка {col} не найдена. Есть: {list(df.columns)}')
    df = df[[cfg.text_column, cfg.label_column]].dropna()
    df[cfg.text_column] = df[cfg.text_column].astype(str).str.strip()
    df = df[df[cfg.text_column] != '']
    df[cfg.label_column] = pd.to_numeric(df[cfg.label_column], errors='coerce')
    df = df.dropna(subset=[cfg.label_column])
    df[cfg.label_column] = df[cfg.label_column].astype(int)

    train_val, test = train_test_split(
        df, test_size=cfg.test_size, random_state=cfg.random_state,
        stratify=df[cfg.label_column])
    rel_val = cfg.val_size / (1 - cfg.test_size)
    train, val = train_test_split(
        train_val, test_size=rel_val, random_state=cfg.random_state,
        stratify=train_val[cfg.label_column])
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

train_df, val_df, test_df = load_and_split(config)

print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')
print()
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    c = split[config.label_column].value_counts().sort_index()
    print(f'  {name:5s}: fake={c.get(0,0)}, real={c.get(1,0)}')

## 3. Подход 1: Zero-shot классификация (NLI)

### Идея подхода

Zero-shot классификация через NLI (Natural Language Inference) позволяет классифицировать тексты **без какого-либо дообучения** на целевом датасете. Модель обучена определять логическую связь между двумя текстами:
- **Premise** (посылка) — текст новости
- **Hypothesis** (гипотеза) — утверждение о достоверности

Модель предсказывает вероятность **entailment** (следование) для каждой гипотезы. Класс с наибольшей вероятностью определяет предсказание.

Используется мультиязычная модель **mDeBERTa-v3**, обученная на 2.7 млн пар NLI на 100+ языках, включая русский.

In [ ]:
# Загрузка модели NLI
nli_tokenizer = AutoTokenizer.from_pretrained(config.nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(config.nli_model_name).to(DEVICE)
nli_model.eval()

print(f'NLI model loaded: {config.nli_model_name}')
print(f'Parameters: {sum(p.numel() for p in nli_model.parameters()) / 1e6:.1f}M')

In [ ]:
@torch.no_grad()
def zero_shot_classify_batch(texts, nli_model, nli_tokenizer, device, max_length=512):
    """Zero-shot классификация через NLI: для каждого текста проверяем
    entailment-скор двух гипотез и выбираем наиболее вероятную."""
    hypotheses = {
        1: 'Это достоверная и правдивая новость.',
        0: 'Это фейковая и ложная новость.'
    }

    predictions = []
    probabilities = []

    for text in tqdm(texts, desc='Zero-shot NLI'):
        scores = {}
        for label, hypothesis in hypotheses.items():
            encoded = nli_tokenizer(
                text[:2048], hypothesis,
                truncation='only_first',
                max_length=max_length,
                return_tensors='pt'
            ).to(device)

            logits = nli_model(**encoded).logits[0]
            probs = torch.softmax(logits, dim=-1)
            # mDeBERTa NLI: [contradiction, neutral, entailment]
            scores[label] = probs[2].item()

        total = scores[0] + scores[1]
        pred = max(scores, key=scores.get)
        conf = scores[pred] / total if total > 0 else 0.5

        predictions.append(pred)
        probabilities.append([1 - conf if pred == 1 else conf, conf if pred == 1 else 1 - conf])

    return predictions, np.array(probabilities)

In [ ]:
# Оценка на тестовой выборке
test_texts = test_df[config.text_column].tolist()
test_labels = test_df[config.label_column].tolist()

nli_preds, nli_probs = zero_shot_classify_batch(
    test_texts, nli_model, nli_tokenizer, DEVICE
)

nli_metrics = {
    'accuracy':  accuracy_score(test_labels, nli_preds),
    'f1':        f1_score(test_labels, nli_preds, average='weighted'),
    'precision': precision_score(test_labels, nli_preds, average='weighted'),
    'recall':    recall_score(test_labels, nli_preds, average='weighted'),
}

print('=== Zero-shot NLI: результаты на тестовой выборке ===')
for k, v in nli_metrics.items():
    print(f'  {k:12s}: {v:.4f}')
print()
print(classification_report(test_labels, nli_preds, target_names=['Фейк', 'Реальная']))

In [ ]:
# Матрица ошибок для NLI
cm_nli = confusion_matrix(test_labels, nli_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_nli, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Фейк', 'Реальная'],
            yticklabels=['Фейк', 'Реальная'], ax=ax)
ax.set_xlabel('Предсказание')
ax.set_ylabel('Истина')
ax.set_title(f'Zero-shot NLI (mDeBERTa) — Accuracy: {nli_metrics["accuracy"]:.3f}')
plt.tight_layout()
plt.show()

In [ ]:
# Освобождаем память от NLI модели
del nli_model, nli_tokenizer
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
print('NLI модель выгружена из памяти')

## 4. Подход 2: Дообучение ruGPT-3 с LoRA

### Идея подхода

**LoRA (Low-Rank Adaptation)** — метод параметрически эффективного дообучения (PEFT), при котором обновляются только небольшие низкоранговые матрицы, добавленные к слоям внимания модели. Это позволяет:

- Обучать **менее 2-3%** от общего числа параметров
- Значительно сократить требования к GPU-памяти
- Сохранить знания предобученной модели (меньше переобучения)
- Хранить компактные адаптеры (несколько МБ вместо сотен МБ)

Используется **ruGPT-3 Small** (`ai-forever/rugpt3small_based_on_gpt2`) — декодерная модель архитектуры GPT-2 (125M параметров), предобученная на русскоязычном корпусе. Поверх неё добавляется голова классификации.

### Ключевое отличие от RuBERT

| | RuBERT | ruGPT-3 + LoRA |
|---|---|---|
| Архитектура | Encoder (BERT) | Decoder (GPT) |
| Предобучение | MLM (маскированный язык) | CLM (авторегрессионный) |
| Дообучение | Все параметры (full fine-tuning) | Только LoRA-адаптеры (PEFT) |
| Обучаемые параметры | ~178M (100%) | ~2-3M (~2%) |

In [ ]:
# Загрузка токенизатора и базовой модели
tokenizer = AutoTokenizer.from_pretrained(config.gpt_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # GPT-2 использует последний токен для классификации

base_model = AutoModelForSequenceClassification.from_pretrained(
    config.gpt_model_name,
    num_labels=config.num_labels,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

total_params = sum(p.numel() for p in base_model.parameters())
print(f'Base model: {config.gpt_model_name}')
print(f'Total parameters: {total_params / 1e6:.1f}M')

In [ ]:
# Настройка LoRA-адаптеров
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=['c_attn', 'c_proj'],  # Слои внимания GPT-2
    bias='none',
)

model = get_peft_model(base_model, lora_config)
model = model.to(DEVICE)

model.print_trainable_parameters()

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, df, tokenizer, text_col, label_col, max_length):
        self.texts = df[text_col].tolist()
        self.labels = df[label_col].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Создание датасетов и загрузчиков
mk = lambda df, shuf: DataLoader(
    NewsDataset(df, tokenizer, config.text_column, config.label_column, config.max_length),
    batch_size=config.batch_size, shuffle=shuf, num_workers=config.num_workers)

train_loader = mk(train_df, True)
val_loader   = mk(val_df, False)
test_loader  = mk(test_df, False)

print(f'Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)

def train_epoch(model, loader, optimizer, scheduler, device, grad_clip):
    model.train()
    total_loss = 0
    preds_all, labels_all = [], []

    for batch in tqdm(loader, desc='Train', leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        preds_all.extend(outputs.logits.argmax(dim=-1).cpu().tolist())
        labels_all.extend(labels.cpu().tolist())

    return {
        'loss': total_loss / len(loader),
        'accuracy': accuracy_score(labels_all, preds_all),
        'f1': f1_score(labels_all, preds_all, average='weighted'),
    }

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    preds_all, labels_all, probs_all = [], [], []

    for batch in tqdm(loader, desc='Eval', leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        total_loss += loss.item()
        probs = torch.softmax(outputs.logits, dim=-1)
        preds_all.extend(probs.argmax(dim=-1).cpu().tolist())
        labels_all.extend(labels.cpu().tolist())
        probs_all.extend(probs.cpu().numpy())

    return {
        'loss':      total_loss / len(loader),
        'accuracy':  accuracy_score(labels_all, preds_all),
        'f1':        f1_score(labels_all, preds_all, average='weighted'),
        'precision': precision_score(labels_all, preds_all, average='weighted'),
        'recall':    recall_score(labels_all, preds_all, average='weighted'),
        'predictions':   preds_all,
        'labels':        labels_all,
        'probabilities': np.array(probs_all),
    }

In [ ]:
# Оптимизатор и планировщик
optimizer = AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

total_steps = len(train_loader) * config.epochs
warmup_steps = int(total_steps * config.warmup_ratio)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'Total steps: {total_steps}, Warmup: {warmup_steps}')

# Обучение
history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
           'val_loss': [], 'val_acc': [], 'val_f1': []}

best_val_f1 = 0.0
best_epoch = 0
no_improve = 0

os.makedirs(os.path.join(config.output_dir, 'rugpt3_lora'), exist_ok=True)

for epoch in range(1, config.epochs + 1):
    print(f'\n{"="*60}')
    print(f'Epoch {epoch}/{config.epochs}')
    print(f'{"="*60}')

    train_res = train_epoch(model, train_loader, optimizer, scheduler, DEVICE, config.grad_clip)
    val_res = evaluate(model, val_loader, DEVICE)

    history['train_loss'].append(train_res['loss'])
    history['train_acc'].append(train_res['accuracy'])
    history['train_f1'].append(train_res['f1'])
    history['val_loss'].append(val_res['loss'])
    history['val_acc'].append(val_res['accuracy'])
    history['val_f1'].append(val_res['f1'])

    print(f'  Train — loss: {train_res["loss"]:.4f}, acc: {train_res["accuracy"]:.4f}, F1: {train_res["f1"]:.4f}')
    print(f'  Val   — loss: {val_res["loss"]:.4f}, acc: {val_res["accuracy"]:.4f}, F1: {val_res["f1"]:.4f}')

    if val_res['f1'] > best_val_f1:
        best_val_f1 = val_res['f1']
        best_epoch = epoch
        no_improve = 0

        model.save_pretrained(os.path.join(config.output_dir, 'rugpt3_lora'))
        tokenizer.save_pretrained(os.path.join(config.output_dir, 'rugpt3_lora'))
        print(f'  >>> Лучшая модель сохранена (val F1: {best_val_f1:.4f})')
    else:
        no_improve += 1
        print(f'  --- Нет улучшения ({no_improve}/{config.patience})')

    if no_improve >= config.patience:
        print(f'\nEarly stopping на эпохе {epoch}. Лучшая эпоха: {best_epoch}')
        break

print(f'\nОбучение завершено. Лучшая эпоха: {best_epoch}, Val F1: {best_val_f1:.4f}')

In [ ]:
# График обучения
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train')
axes[0].plot(epochs_range, history['val_loss'], 's-', label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], 'o-', label='Train')
axes[1].plot(epochs_range, history['val_acc'], 's-', label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(epochs_range, history['train_f1'], 'o-', label='Train')
axes[2].plot(epochs_range, history['val_f1'], 's-', label='Val')
axes[2].set_title('F1-score'); axes[2].set_xlabel('Epoch'); axes[2].legend()

fig.suptitle('ruGPT-3 + LoRA: история обучения', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Загружаем лучшую модель и оцениваем на тестовой выборке
best_base = AutoModelForSequenceClassification.from_pretrained(
    config.gpt_model_name, num_labels=config.num_labels
)
best_base.config.pad_token_id = tokenizer.pad_token_id
best_model = PeftModel.from_pretrained(best_base, os.path.join(config.output_dir, 'rugpt3_lora'))
best_model = best_model.to(DEVICE)
best_model.eval()

test_res = evaluate(best_model, test_loader, DEVICE)

lora_metrics = {
    'accuracy':  test_res['accuracy'],
    'f1':        test_res['f1'],
    'precision': test_res['precision'],
    'recall':    test_res['recall'],
}

print('=== ruGPT-3 + LoRA: результаты на тестовой выборке ===')
for k, v in lora_metrics.items():
    print(f'  {k:12s}: {v:.4f}')
print()
print(classification_report(
    test_res['labels'], test_res['predictions'],
    target_names=['Фейк', 'Реальная']
))

In [ ]:
# Матрица ошибок для LoRA
cm_lora = confusion_matrix(test_res['labels'], test_res['predictions'])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_lora, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Фейк', 'Реальная'],
            yticklabels=['Фейк', 'Реальная'], ax=ax)
ax.set_xlabel('Предсказание')
ax.set_ylabel('Истина')
ax.set_title(f'ruGPT-3 + LoRA — Accuracy: {lora_metrics["accuracy"]:.3f}')
plt.tight_layout()
plt.show()

## 5. Сравнение всех подходов

Сравниваем LLM-подходы с ранее обученными моделями: классическими (TF-IDF, Word2Vec) и RuBERT.

In [ ]:
# Загружаем метрики ранее обученных моделей
with open('../../models/rubert/metrics.json', 'r') as f:
    rubert_m = json.load(f)['rubert']
with open('../../results/metrics/metrics_tfidf_tuned.json', 'r') as f:
    tfidf_m = json.load(f)
with open('../../results/metrics/metrics_w2v.json', 'r') as f:
    w2v_m = json.load(f)

comparison = pd.DataFrame([
    {'Модель': 'LR (Word2Vec)',        'Тип': 'Classical', 'Accuracy': w2v_m['logisticregression']['val_accuracy'],
     'F1': w2v_m['logisticregression']['val_f1']},
    {'Модель': 'RF (Word2Vec)',         'Тип': 'Classical', 'Accuracy': w2v_m['randomforest']['val_accuracy'],
     'F1': w2v_m['randomforest']['val_f1']},
    {'Модель': 'Naive Bayes (TF-IDF)',  'Тип': 'Classical', 'Accuracy': tfidf_m['naive_bayes']['val_acc'],
     'F1': tfidf_m['naive_bayes']['val_f1']},
    {'Модель': 'RF (TF-IDF)',           'Тип': 'Classical', 'Accuracy': tfidf_m['random_forest']['val_acc'],
     'F1': tfidf_m['random_forest']['val_f1']},
    {'Модель': 'LR (TF-IDF)',           'Тип': 'Classical', 'Accuracy': tfidf_m['logistic_regression']['val_acc'],
     'F1': tfidf_m['logistic_regression']['val_f1']},
    {'Модель': 'RuBERT (full FT)',      'Тип': 'Transformer', 'Accuracy': rubert_m['test_acc'],
     'F1': rubert_m['test_f1'],         'Precision': rubert_m['test_precision'], 'Recall': rubert_m['test_recall']},
    {'Модель': 'Zero-shot NLI (mDeBERTa)', 'Тип': 'LLM', 'Accuracy': nli_metrics['accuracy'],
     'F1': nli_metrics['f1'],           'Precision': nli_metrics['precision'], 'Recall': nli_metrics['recall']},
    {'Модель': 'ruGPT-3 + LoRA',       'Тип': 'LLM', 'Accuracy': lora_metrics['accuracy'],
     'F1': lora_metrics['f1'],          'Precision': lora_metrics['precision'], 'Recall': lora_metrics['recall']},
])

comparison = comparison.sort_values('Accuracy', ascending=True).reset_index(drop=True)
comparison.index = range(1, len(comparison) + 1)
print(comparison.to_string())

In [ ]:
# Визуализация сравнения
colors = {'Classical': '#3498db', 'Transformer': '#e74c3c', 'LLM': '#2ecc71'}
bar_colors = [colors[t] for t in comparison['Тип']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, metric in enumerate(['Accuracy', 'F1']):
    ax = axes[i]
    bars = ax.barh(comparison['Модель'], comparison[metric], color=bar_colors, edgecolor='white', height=0.6)
    ax.set_xlim(0.8, 1.0)
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.set_xlabel(metric)

    for bar, val in zip(bars, comparison[metric]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=10)

# Легенда
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in colors.items()]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, fontsize=11,
           bbox_to_anchor=(0.5, 1.02))

fig.suptitle('Сравнение всех подходов к детекции фейковых новостей', fontsize=15, fontweight='bold', y=1.06)
plt.tight_layout()
plt.savefig('../../assets/llm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Матрицы ошибок: NLI vs LoRA
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, cm, title, cmap in [
    (axes[0], cm_nli,  f'Zero-shot NLI\nAccuracy: {nli_metrics["accuracy"]:.3f}',  'Blues'),
    (axes[1], cm_lora, f'ruGPT-3 + LoRA\nAccuracy: {lora_metrics["accuracy"]:.3f}', 'Purples'),
]:
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=['Фейк', 'Реальная'],
                yticklabels=['Фейк', 'Реальная'], ax=ax)
    ax.set_xlabel('Предсказание')
    ax.set_ylabel('Истина')
    ax.set_title(title)

fig.suptitle('Матрицы ошибок LLM-подходов', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../assets/llm_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Выводы

### Основные результаты

1. **Zero-shot NLI** демонстрирует способность больших предобученных моделей классифицировать фейковые новости **без какого-либо дообучения** на целевом датасете. Это важный baseline, показывающий общие знания модели о языке и фактах.

2. **ruGPT-3 + LoRA** показывает, что декодерная архитектура (GPT) может быть эффективно адаптирована для задачи классификации с помощью параметрически эффективного дообучения, при этом обучая менее 3% параметров модели.

3. Сравнение с RuBERT и классическими моделями позволяет объективно оценить вклад различных архитектурных решений и методов обучения.

### Практическая значимость

- **LoRA-адаптеры** занимают всего несколько МБ, что делает их удобными для развёртывания и хранения
- **Zero-shot подход** может использоваться как быстрый baseline без необходимости обучения
- Комбинация нескольких подходов (ансамбль) может дать ещё более надёжные результаты

In [ ]:
# Сохранение всех артефактов
os.makedirs(config.output_dir, exist_ok=True)

# Метрики
all_metrics = {
    'zero_shot_nli': nli_metrics,
    'rugpt3_lora': {
        **lora_metrics,
        'best_epoch': best_epoch,
        'best_val_f1': best_val_f1,
    },
    'config': asdict(config),
}

with open(os.path.join(config.output_dir, 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=4, ensure_ascii=False)

# История обучения
with open(os.path.join(config.output_dir, 'history.json'), 'w') as f:
    json.dump(history, f, indent=4)

# Предсказания на тестовой выборке
pred_df = test_df[[config.text_column, config.label_column]].copy()
pred_df['nli_pred'] = nli_preds
pred_df['nli_prob_real'] = nli_probs[:, 1]
pred_df['lora_pred'] = test_res['predictions']
pred_df['lora_prob_real'] = test_res['probabilities'][:, 1]
pred_df.to_csv(os.path.join(config.output_dir, 'test_predictions.csv'), index=False)

# Таблица сравнения
comparison.to_csv('../../assets/llm_comparison.csv', index=False)

print('Сохранено:')
for f in ['metrics.json', 'history.json', 'test_predictions.csv', 'rugpt3_lora/']:
    path = os.path.join(config.output_dir, f)
    if os.path.exists(path):
        if os.path.isdir(path):
            size = sum(os.path.getsize(os.path.join(path, x)) for x in os.listdir(path)) / 1024**2
        else:
            size = os.path.getsize(path) / 1024**2
        print(f'  {path} ({size:.2f} MB)')
print(f'  assets/llm_comparison.png')
print(f'  assets/llm_confusion_matrices.png')
print(f'  assets/llm_comparison.csv')
print('\nГотово!')